### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import tomllib
import numpy as np
import bambi as bmb
import arviz as az
from scipy.special import expit

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'

sync_at_file = os.path.join(sim_results_folder, 'highres_arnold_tongues.npy')
fr_at_file = os.path.join(sim_results_folder, 'highres_firing_rates.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [4]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)[0][:,::6, ::6]
sync_results_vector = sync_results.mean(axis=0).flatten()

fr_results = np.load(fr_at_file)[0]
#fr_results_vector = fr_results.mean(axis=0).flatten()
fr_results = np.load(fr_at_file)[0]
fr_results = fr_results.mean(axis=0)
fr_results -= np.outer(fr_results[:,-1], np.ones(5))
fr_results_vector = fr_results.flatten()

data = load_data(emp_at_file)
# Get the data for the first session
data = get_session_data(data, 1)

# Map synchrony values to each condition in DataFrame
data['Synchrony'] = data['Condition'].apply(lambda x: sync_results_vector[x-1])
data['FiringRate'] = data['Condition'].apply(lambda x: fr_results_vector[x-1])

data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony', 'FiringRate'])

### Mediation and model comparison

In [5]:
# Features-only hierarchical logistic regression
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1|SubjectID)",
    data=data,
    family="bernoulli"
)

idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

# Mechanism A (Synchrony)
model_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + ContrastHeterogeneity * GridCoarseness + (1|SubjectID) + (0 + Synchrony|SubjectID)",
    data=data,
    family="bernoulli"
)
idata_sync = model_sync.fit(draws=2000, tune=2000, target_accept=0.95, idata_kwargs={"log_likelihood": True}, progressbar=False)


# Mechanism B (Firing Rate)
model_fr = bmb.Model(
    "Correct ~ 1 + FiringRate + ContrastHeterogeneity * GridCoarseness + (1|SubjectID) + (0 + FiringRate|SubjectID)",
    data=data,
    family="bernoulli"
)
idata_fr = model_fr.fit(draws=2000, tune=2000, target_accept=0.95, idata_kwargs={"log_likelihood": True}, progressbar=False)

# Compare models (LOO)
az.compare({
    "stimulus features": idata_features,
    "synchrony": idata_sync,
    "firing rate": idata_fr
}, method="BB-pseudo-BMA")


WARNING (pytensor.link.c.cmodule): Deleting (Broken cache directory [EOF]): /home/mario/.pytensor/compiledir_Linux-6.8--generic-x86_64-with-glibc2.35-x86_64-3.11.13-64/tmpuh5apscz
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 54 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
synchrony,0,-3586.416840,18.436947,0.000000,9.827091e-01,29.815990,0.000000,False,log
firing rate,1,-3603.731469,17.899362,17.314629,1.729095e-02,29.510220,8.464477,False,log
stimulus features,2,-3634.171186,10.953478,47.754346,2.300671e-11,29.569096,9.612487,False,log


In [18]:
az.summary(idata_features, var_names=["Intercept", "ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"], hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.656,0.153,0.362,0.964,0.004,0.004,1600.0,1975.0,1.0
ContrastHeterogeneity,-0.546,0.029,-0.603,-0.489,0.000,0.000,5562.0,5147.0,1.0
GridCoarseness,-0.241,0.030,-0.300,-0.183,0.000,0.000,5661.0,4045.0,1.0
ContrastHeterogeneity:GridCoarseness,0.206,0.030,0.149,0.264,0.000,0.000,5650.0,4640.0,1.0


In [21]:
az.summary(idata_sync, var_names=["Intercept", "Synchrony", "ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"], hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.726,0.194,0.328,1.108,0.004,0.004,2254.0,2815.0,1.0
Synchrony,0.193,0.189,-0.206,0.552,0.004,0.003,2802.0,3687.0,1.0
ContrastHeterogeneity,-0.464,0.061,-0.584,-0.346,0.001,0.001,5749.0,5609.0,1.0
GridCoarseness,-0.231,0.035,-0.297,-0.163,0.000,0.000,5861.0,6054.0,1.0
ContrastHeterogeneity:GridCoarseness,0.201,0.035,0.134,0.273,0.000,0.000,5745.0,5508.0,1.0


# Main analysis (effect of synchrony)

In [6]:
pure_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + (1|SubjectID) + (0 + Synchrony|SubjectID)",
    data=data,
    family="bernoulli"
)
idata_pure = pure_sync.fit(draws=2000, tune=2000, target_accept=0.95, idata_kwargs={"log_likelihood": True}, progressbar=False)

beta_sync = idata_pure.posterior["Synchrony"].values.flatten()
prob_positive = np.mean(beta_sync > 0)

or_sync_mean = np.exp(beta_sync).mean()
or_sync_low = np.percentile(np.exp(beta_sync), 2.5)
or_sync_high = np.percentile(np.exp(beta_sync), 97.5)

print(f'P(Synchrony > 0) = {prob_positive:.3f}')
az.summary(idata_pure, var_names=["Intercept", "Synchrony"], hdi_prob=0.95)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 121 seconds.


P(Synchrony > 0) = 0.998


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.714,0.190,0.351,1.113,0.004,0.004,1997.0,2529.0,1.0
Synchrony,0.671,0.182,0.316,1.060,0.004,0.004,1978.0,2691.0,1.0


In [7]:
beta0 = idata_pure.posterior["Intercept"].values.flatten()

# Predicted accuracy at -1 SD and +1 SD synchrony
p_low = expit(beta0 - beta_sync)
p_high = expit(beta0 + beta_sync)


table = PrettyTable()
table.field_names = ["Metric", "Value"]
table.add_row(["Probability low synchrony", p_low.mean()])
table.add_row(["Probability high synchrony", p_high.mean()])
table.add_row(["Difference", (p_high - p_low).mean()])

print(table)

+----------------------------+---------------------+
|           Metric           |        Value        |
+----------------------------+---------------------+
| Probability low synchrony  |  0.5106424890254068 |
| Probability high synchrony |  0.7963542949377581 |
|         Difference         | 0.28571180591235146 |
+----------------------------+---------------------+


### Sensitivity Analysis

In [8]:
with open('../config/simulation/simulation.toml', 'rb') as f:
    sim_config = tomllib.load(f)

seed = sim_config['random_seed']
rng = np.random.default_rng(seed)

In [15]:
num_repetitions = 100
effect_sizes = np.linspace(0.3, 0.9, 7) # log-odds

formula = "Correct ~ 1 + Synchrony + (1|SubjectID) + (0 + Synchrony|SubjectID)"
effect_of_interest = "Synchrony"

In [16]:
subject_index, num_subjects = create_subject_index(data)
synchrony = data["Synchrony"].to_numpy()

# Extract posteriors for intercept and beta_sync as well as for the subject-level random effects
posteriors = az.extract(idata_pure, combined=True)

intercept = np.median(posteriors["Intercept"].to_numpy())
beta_sync = np.median(posteriors["Synchrony"].to_numpy())

sdev_intercept = np.median(posteriors["1|SubjectID_sigma"].to_numpy())              # random intercept SD
sdev_beta_sync = np.median(posteriors["Synchrony|SubjectID_sigma"].to_numpy())    # random slope SD


results = []
table = PrettyTable()
table.field_names = ["Effect Size (log-odds)", "Detection Rate"]
for effect_size in effect_sizes:
    detections = 0
    for r in range(num_repetitions):
        simulated_df = simulate_correct(data, effect_size, synchrony, intercept, sdev_beta_sync, sdev_intercept, subject_index, num_subjects, rng)
        detected = fit_and_decide(simulated_df, formula, effect_of_interest, draws=1000,
                   tune=1000, threshold=0.975)
        detections += int(detected)
    detection_rate = detections / num_repetitions
    results.append({"true_beta_sync": effect_size, "detect_rate": detection_rate})
    table.add_row([effect_size, detection_rate])

print(table)


Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 39 seconds.
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 41 seconds.
Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 1_000 tune and

+------------------------+----------------+
| Effect Size (log-odds) | Detection Rate |
+------------------------+----------------+
|          0.3           |      0.31      |
|          0.4           |      0.55      |
|          0.5           |      0.71      |
|   0.6000000000000001   |      0.83      |
|   0.7000000000000001   |      0.95      |
|          0.8           |      0.99      |
|          0.9           |      0.96      |
+------------------------+----------------+
